# Sloshing tank (SPHERIC Test Case 10)

A shallow layer of water (depth `h = 0.093 m`) in a rigid rectangular tank
(`0.900 x 0.508 m`) that is **rolled** harmonically about the centre of its
floor (amplitude ~ +-4 deg, period ~ 1.7 s, close to the first sloshing mode).
The figure of merit is the pressure history at wall **Sensor 1**, on the left
wall at the still-water line `(-0.45, 0.093)` (see `SPHERIC_TestCase10/SPHERIC_TestCase10_Fig1.png`).

This notebook is the interactive companion of `run_sloshingTank.py` and the
`warpSPH.cases.sloshingTank` case; it ports diffSPH's
`examples/weaklyCompressible/16_SloshingTank.ipynb`. See `PLAN.md` for the full
write-up **and the results** (`RESULTS` section) -- read the *Findings* cell at
the bottom first if you just want the verdict.

## Method: rotate gravity in the tank frame

Rather than move the walls, the run is solved in the frame that rolls with the
tank and **gravity is rotated** by the roll angle `theta(t)`:

```
g_dir(t) = R(-theta(t)) . (0, -1) = (-sin theta, -cos theta)
```

`modules/gravity/directional.py` re-reads `gravityConfig.direction` every step,
so a `postStep` hook rewrites it from a spline of the tabulated roll angle
(`SPHERIC_TestCase10/data_files/lateral_water_1x.txt`). The non-inertial Euler
and centrifugal terms are dropped -- small at this amplitude/period, and
dropped by the diffSPH reference too.

## Two schemes

- **WCSPH** (`scheme='deltaSPH'`, weakly compressible): sensor pressure from the
  nearest boundary particle's density via the Tait EOS, scaled to Pa by
  `rho0Physical` (the scheme runs at `restDensity = 1`). *Result: tracks the
  wave timing, then the first impact destabilises it and it diverges at
  `t ~ 3.4 s`.*
- **DFSPH** (`scheme='divergenceFree'`, incompressible): the VD+PS scheme
  enforces incompressibility with a position shift and **carries no stored
  pressure field**, so `state.pressures` is all zeros and there is no Sensor-1
  signal. It also has a known quiescent free-surface-under-gravity weakness
  (`DFSPH_IMPROVEMENT_PLAN.md` Part 23 / `hydrostaticColumn`). *Result: runs to
  7 s but is not evaluable on the figure of merit; the free surface collapses
  to ~0.5 rho0.*


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.autonotebook import tqdm

from warpSPH import *
from warpSPH.cases.sloshingTank import sloshingTankCase, loadRollHistory, SLOSHING_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext

SPHERIC = 'SPHERIC_TestCase10'
ROLL_FILE = os.path.join(SPHERIC, 'data_files', 'lateral_water_1x.txt')
os.makedirs('output', exist_ok=True)

In [ ]:
# Every knob, made explicit. `sloshingTankCase.defaults` / `.params` are the
# same values `run_sloshingTank.py` starts from.
spec = CaseSpec(caseName=sloshingTankCase.name, scheme=sloshingTankCase.scheme,
                params=dict(sloshingTankCase.params)).merged(**sloshingTankCase.defaults)

SCHEME = 'wcsph'   # 'wcsph' -> deltaSPH   |   'dfsph' -> divergenceFree

if SCHEME == 'wcsph':
    spec = spec.merged(scheme='deltaSPH')
else:
    spec = spec.merged(scheme='divergenceFree', integrationScheme='semiImplicitEuler',
                       kernel='Wendland2', supportMode='SuperSymmetric',
                       cflFactor=0.2, dt=1e-3, maxDt=2e-3)

spec = spec.merged(
    nx=120,            # particles across the 0.9 m width; dx = 0.9 / nx
    tLimit=7.0,        # full record runs to 8.35 s; the impact window is ~2-7 s
    caseName=f'16-sloshingTank-{SCHEME}',
    plot=True, plotInterval=100,
)
spec

In [ ]:
# Build the run with the real case code -- nothing re-derived here.
ctx = buildContext(sloshingTankCase, spec)
sloshingTankCase.configureScheme(ctx)
system = sloshingTankCase.buildSystem(ctx)
sloshingTankCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

kinds = runningState.state.kinds
dx = ctx.config.dx
print(f'scheme   : {ctx.scheme.name}')
print(f'dt       : {float(ctx.config.dt):.3e} s'
      + (f'   c_s = {ctx.schemeConfig.fluid.fixedSoundSpeed:.2f} m/s' if SCHEME == 'wcsph' else ''))
print(f'dx       : {dx:.4f} m   ({ctx.param("fillDepth") / dx:.1f} particles deep)')
print(f'particles: {len(runningState.state.positions)}  '
      f'({int((kinds == 0).sum())} fluid, {int((kinds != 0).sum())} boundary)')

# Locate Sensor 1 (nearest boundary particle to sensorPos), same as the case.
sensorPos = torch.as_tensor(ctx.param('sensorPos'), device=runningState.state.positions.device,
                            dtype=runningState.state.positions.dtype)
bnd = runningState.state.kinds == 1
bIdx = torch.arange(len(bnd), device=bnd.device)[bnd]
sensorIdx = int(bIdx[torch.argmin(torch.linalg.norm(
    runningState.state.positions[bnd] - sensorPos, dim=-1))].item())
sensorActual = runningState.state.positions[sensorIdx].detach().cpu().numpy()
print(f'Sensor 1 target {ctx.param("sensorPos")}  ->  particle at {sensorActual.round(4).tolist()}')

In [ ]:
# What was built: fluid + wall regions, the sensor, the still-water line.
fig, axis = plt.subplots(1, 1, figsize=(11, 6), squeeze=False)
plotRegions(ctx.schemeConfig.regions, axis[0, 0], plotFluid=True, plotParticles=True)
d = ctx.config.domain
axis[0, 0].axhline(ctx.param('fillDepth'), color='red', ls='--', alpha=0.4, label='still-water line')
axis[0, 0].scatter(*sensorActual, s=80, c='k', marker='x', label='Sensor 1', zorder=5)
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(d.min[0].item(), d.max[0].item())
axis[0, 0].set_ylim(d.min[1].item(), d.max[1].item())
axis[0, 0].legend(loc='upper right')
axis[0, 0].set_title(f'{len(ctx.schemeConfig.regions)} regions, '
                     f'{len(runningState.state.positions)} particles')
fig.tight_layout()

In [ ]:
# The prescribed motion and the measured Sensor-1 signal.
roll = loadRollHistory(ROLL_FILE)
expT = roll['t']; expRollDeg = np.degrees(roll['theta']); expPPa = roll['pressureMbar'] * 100.0

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(expT, expPPa, lw=0.8, color='0.4')
ax[0].set_ylabel('measured p [Pa]'); ax[0].grid(alpha=0.3)
ax[0].set_title('SPHERIC TC10 -- prescribed roll and measured Sensor 1 pressure')
ax[1].plot(expT, expRollDeg, lw=0.8, color='tab:red')
ax[1].set_ylabel('roll [deg]'); ax[1].set_xlabel('time [s]'); ax[1].grid(alpha=0.3)
ax[1].set_xlim(0, spec.tLimit)
fig.tight_layout()

In [ ]:
# Live field plot (velocity + density). `buildFieldPlotter` / `refreshFieldPlotter`
# are the live-updating path in Jupyter (the case's setupPlot/updatePlot go
# through openWindow, which does not tick inside a cell).
plotter = buildFieldPlotter(ctx, runningState, SLOSHING_FIELDS, figsize=(13, 5)) if spec.plot else None

In [ ]:
# The step loop, unrolled -- the same call `warpSPH.runner.runner._run` makes,
# with the roll-driven gravity update (`postStep`) and the sensor recording
# (`diagnostics`) visible at the hook points.
trajectory = [dict(sloshingTankCase.diagnostics(ctx, runningState), step=-1, t=0.0)]

i = 0
with tqdm(total=1000) as bar:
    while True:
        stepResult = ctx.integrator.function(
            state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
            config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False)
        runningState = stepResult.state
        sloshingTankCase.postStep(ctx, runningState, i)          # <- roll -> gravity dir
        ctx.config.dt = sloshingTankCase.timestep(ctx, runningState)

        t = float(runningState.t)
        row = dict(sloshingTankCase.diagnostics(ctx, runningState), step=i, t=t)  # <- sensor
        trajectory.append(row)
        bar.n = min(1000, int(t / spec.tLimit * 1000))
        bar.set_description(f"t={t:.3f} roll={row['rollAngleDeg']:+.2f}deg "
                            f"p={row['sensorPressure']:+.0f}Pa")
        bar.refresh()

        if plotter is not None and (i % spec.plotInterval == 0):
            refreshFieldPlotter(ctx, runningState, plotter, SLOSHING_FIELDS, step=i)

        if torch.any(~torch.isfinite(runningState.state.velocities)):
            print(f'non-finite velocities at step {i}; stopping.'); break
        if t >= spec.tLimit:
            break
        i += 1

In [ ]:
# Simulated vs measured Sensor-1 pressure.
from scipy.ndimage import gaussian_filter1d
t = np.array([r['t'] for r in trajectory])
p = np.array([r['sensorPressure'] for r in trajectory])
probe = np.array([r.get('sensorPressureProbe', np.nan) for r in trajectory])
rollApplied = np.array([r['rollAngleDeg'] for r in trajectory])

dt = np.median(np.diff(t)); grid = np.arange(t[1], t[-1], dt)
smooth = gaussian_filter1d(np.interp(grid, t, p), 0.01 / dt)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw=dict(height_ratios=[3, 1]))
ax[0].plot(expT, expPPa, lw=1.0, ls='--', color='0.4', label='measured (Sensor 1)')
ax[0].plot(t, p, lw=0.5, color='tab:blue', alpha=0.35, label='simulated (raw)')
ax[0].plot(grid, smooth, lw=1.6, color='tab:blue', label='simulated (smoothed 10 ms)')
ax[0].plot(t, probe, lw=0.8, color='tab:green', alpha=0.6, label='simulated (fluid-probe)')
ax[0].set_ylabel('Sensor 1 pressure [Pa]'); ax[0].grid(alpha=0.3)
ax[0].legend(fontsize=8); ax[0].set_title(f'Sloshing tank -- {SCHEME}, nx={spec.nx}')
ax[1].plot(expT, expRollDeg, lw=1.0, ls='--', color='0.4', label='prescribed')
ax[1].plot(t, rollApplied, lw=1.0, color='tab:red', label='applied')
ax[1].set_ylabel('roll [deg]'); ax[1].set_xlabel('time [s]'); ax[1].grid(alpha=0.3)
ax[1].legend(fontsize=8); ax[1].set_xlim(0, spec.tLimit)
fig.tight_layout()
fig.savefig(f'output/{SCHEME}_sensor_pressure_notebook.png', dpi=150)

In [ ]:
# Health check: density bounds (+-1% for WCSPH) and kinetic energy.
rhoMin = np.array([r['minDensity'] for r in trajectory])
rhoMax = np.array([r['maxDensity'] for r in trajectory])
ke = np.array([r['kineticEnergy'] for r in trajectory])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t, rhoMax, label='max'); ax[0].plot(t, rhoMin, label='min')
ax[0].axhspan(0.99, 1.01, color='green', alpha=0.1, label='+-1%')
ax[0].set_xlabel('t [s]'); ax[0].set_ylabel(r'$\rho / \rho_0$'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(t, ke); ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('kinetic energy'); ax[1].grid(alpha=0.3)
fig.tight_layout()

## Findings (nx = 100, `run_sloshingTank.py`, `t -> 7 s`)

Full write-up + figures: `PLAN.md` (`RESULTS` / `FOLLOW-UP`) and
`output/{wcsph,dfsph}_sensor_pressure.pdf`.

**The case wiring is correct.** Applied roll overlays prescribed exactly;
under both schemes the sensor first responds at `t ~ 2.35 s`, the measured
first-impact time, with a matching small pre-impact bump.

**DFSPH (`divergenceFree`) — reproduces the signal, with a phase artifact.**
Once `schemes/dfsph.py` was wired to persist its solve pressures
(`pressures` = constant-density solve, `soundspeeds` = divergence-free solve),
the Sensor-1 curve matches well: **first impact ~3.6 kPa sim vs ~3.6 kPa
measured**, inside the 2.2–13 kPa repeatability band. But the simulated wave
**arrives progressively earlier**: cross-correlation lag −31 ms (cycle 1) →
−37 ms (cycle 2) → −78 ms (cycle 3), i.e. ~2 % → ~5 % of the ~1.64 s period.
This is the known SPH free-surface wave-celerity / dispersion error,
accumulating because the case is *resonant* (roll period ≈ first sloshing
mode) and aggravated by the DFSPH free surface degrading over the run
(`minDensity → 0.48`) and the coarse ~10-particle depth. Late impacts also
overshoot (~12 kPa on the 3rd) as the surface density collapses. A resolution
study (nx 150/200) is the next check on the phase drift.

**WCSPH (`deltaSPH`) — diverges on the first slam.** Clean while quiescent
(ρ within 0.3 %); the first wave impact at `t ~ 2.35 s` drives a pressure
spike to ~5.7 kPa smoothed / ~50 kPa raw (18 % local density excursion,
Ma ≈ 0.42), an undamped ±10 kPa ring-down, and hard divergence at `t = 3.41 s`.
`c_s = 16.6` (vs diffSPH's hardcoded 20) is marginal *for the impact*. A
stiffer re-run — `--targetDt 1e-4 --alpha 0.06` → `c_s ≈ 33` — is in
`output/wcsph_hics/`.

**Verdict.** The tooling reproduces SPHERIC TC10 faithfully. **DFSPH now tracks
the pressure signal** (first impact on the money; wave phase runs 2–5 % fast —
a known SPH celerity artifact). **WCSPH** still diverges at the first impact
at this `c_s`/dissipation.
